In [ ]:
#Load IOS feature selected dataset for modeling:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, f1_score, accuracy_score
from imblearn.over_sampling import SMOTE
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder



iOS_selected_features = pd.read_csv("../Processed_Data/Selected_FeaturesDatasets/iOS_SelectedFeatures.csv")

In [ ]:
"stress" in iOS_selected_features.columns

In [ ]:
#Preparing dataset for analysis:
#Select relevant columns for analysis:
y = iOS_selected_features['stress']
X = iOS_selected_features.drop(columns=['stress', 'uid', 'day'])  # Drop target, ID columns, and date
print("Final feature set columns:", X.columns)
print("Final feature set shape:", X.shape)  

In [ ]:
#Splitting data into test and train sets to prevent data leakage:

#Column that identifies groups (participants):
group_col = 'uid'

#80/20 training testing split, with one testing group:
gss = GroupShuffleSplit(test_size=0.2, n_splits=1, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=iOS_selected_features[group_col]))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
groups_train = iOS_selected_features[group_col].iloc[train_idx]

#Checking stress label distribution to ensure the groups are stratified:
print("Train distribution:")
print(y_train.value_counts(normalize=True))

print("\nTest distribution:")
print(y_test.value_counts(normalize=True))

In [ ]:
# Stratified Group K-Fold
skf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

**Random Forest Classifier**

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=1000,
    max_depth=10,                # Shallower to prevent "memorizing" fold noise
    min_samples_split=40,        # Higher threshold to ensure patterns are statistically significant
    min_samples_leaf=15,         # Ensures every "leaf" is a group, not an outlier
    max_features=0.5,            # Look at half the features for each split
    max_samples=0.6,             # Each tree sees a different 60% slice of data
    class_weight='balanced',     # Helps minority classes without the volatility of subsample-weighting
    random_state=42,
    n_jobs=-1,
    ccp_alpha=0.0005             # Final pruning touch
)

# CROSS-VALIDATION
skf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
RF_fold_f1, RF_fold_acc = [], []

print("--- STARTING CV ---")

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train, groups=groups_train)):
    X_fold_train = X_train.iloc[train_idx]
    y_fold_train = y_train.iloc[train_idx]
    X_fold_val = X_train.iloc[val_idx]
    y_fold_val = y_train.iloc[val_idx]
    
    imputer = SimpleImputer(strategy='median')
    X_fold_train_imp = imputer.fit_transform(X_fold_train)
    X_fold_val_imp = imputer.transform(X_fold_val)
    
    rf_model.fit(X_fold_train_imp, y_fold_train)
    val_preds = rf_model.predict(X_fold_val_imp)
    
    f1 = f1_score(y_fold_val, val_preds, average='weighted')
    acc = accuracy_score(y_fold_val, val_preds)
    RF_fold_f1.append(f1); RF_fold_acc.append(acc)
    print(f"Fold {fold+1} | F1: {f1:.4f} | Acc: {acc:.4f}")

print(f"\nAverage RF F1: {np.mean(RF_fold_f1):.4f}")

# FINAL TRAINING
final_imputer = SimpleImputer(strategy='median')
X_train_final = final_imputer.fit_transform(X_train)
X_test_final = final_imputer.transform(X_test)

rf_model.fit(X_train_final, y_train)
test_preds = rf_model.predict(X_test_final)

print("\n TEST PERFORMANCE ")
print(f"Accuracy: {accuracy_score(y_test, test_preds):.4f}")
print(classification_report(y_test, test_preds))

**XGBoost Classifier**

In [ ]:
from sklearn.utils.class_weight import compute_sample_weight

xgb_params = {
    'n_estimators': 1500,
    'max_depth': 4,              
    'learning_rate': 0.01,       
    'subsample': 0.7,
    'colsample_bytree': 0.6,
    'colsample_bynode': 0.5,     
    'objective': 'multi:softmax',
    'num_class': 5,
    'gamma': 2,                  
    'reg_lambda': 5,             
    'random_state': 42,
    'n_jobs': -1,
    'tree_method': 'hist',
    'eval_metric': 'mlogloss',
    'early_stopping_rounds': 50
}

skf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

XGB_fold_f1 = []
XGB_fold_acc = []
best_iterations = []

print("--- STARTING CV ---")

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train, groups=groups_train)):
    # Prepare Fold Data
    X_fold_train = X_train.iloc[train_idx].select_dtypes(include=[np.number])
    X_fold_val = X_train.iloc[val_idx].select_dtypes(include=[np.number])
    
    y_fold_train_adj = y_train.iloc[train_idx] - 1
    y_fold_val_adj = y_train.iloc[val_idx] - 1
    
    # GENTLE CLASS WEIGHTING
    weights = compute_sample_weight(class_weight='balanced', y=y_fold_train_adj)
    weights = np.sqrt(weights) 
    
    fold_model = XGBClassifier(**xgb_params)
    
    fold_model.fit(
        X_fold_train, y_fold_train_adj,
        eval_set=[(X_fold_val, y_fold_val_adj)],
        sample_weight=weights,
        verbose=False
    )
    
    # Store results
    best_iterations.append(fold_model.best_iteration)
    val_preds = fold_model.predict(X_fold_val) + 1
    
    f1 = f1_score(y_train.iloc[val_idx], val_preds, average='weighted')
    acc = accuracy_score(y_train.iloc[val_idx], val_preds)
    
    XGB_fold_f1.append(f1)
    XGB_fold_acc.append(acc)
    
    print(f"Fold {fold+1} | Trees: {fold_model.best_iteration} | F1: {f1:.4f} | Acc: {acc:.4f}")

# 2. FINAL TRAINING ON FULL DATA
avg_trees = int(np.mean(best_iterations))
print(f"\nAverage Trees: {avg_trees}")

X_train_final = X_train.select_dtypes(include=[np.number])
X_test_final = X_test.select_dtypes(include=[np.number])

final_weights = np.sqrt(compute_sample_weight(class_weight='balanced', y=y_train - 1))

# Final model parameters (minus early stopping keys)
final_params = {k: v for k, v in xgb_params.items() if k not in ['early_stopping_rounds', 'eval_metric', 'n_estimators' ]}
final_model = XGBClassifier(**final_params, n_estimators=avg_trees)

final_model.fit(X_train_final, y_train - 1, sample_weight=final_weights)

# 3. TEST SET PERFORMANCE
test_preds = final_model.predict(X_test_final) + 1

print("\n--- STABILIZED TEST PERFORMANCE ---")
print(f"Accuracy: {accuracy_score(y_test, test_preds):.4f}")
print(f"F1 Score: {f1_score(y_test, test_preds, average='weighted'):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, test_preds))

**Multinomial Logistic Regression**

In [ ]:
lr_model = LogisticRegression(
    solver='lbfgs',
    max_iter=1000,
    class_weight=None,  
    random_state=42
)

LR_fold_f1 = []
LR_fold_acc = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train, groups=groups_train)):
    
    # Split + numeric only
    X_fold_train = X_train.iloc[train_idx].select_dtypes(include=[np.number])
    y_fold_train = y_train.iloc[train_idx]
    X_fold_val = X_train.iloc[val_idx].select_dtypes(include=[np.number])
    y_fold_val = y_train.iloc[val_idx]
    
    # Impute
    imputer = SimpleImputer(strategy='median')
    X_fold_train = imputer.fit_transform(X_fold_train)
    X_fold_val = imputer.transform(X_fold_val)
    
    # Scale ( to avoid domination by features with larger ranges)
    scaler = StandardScaler()
    X_fold_train = scaler.fit_transform(X_fold_train)
    X_fold_val = scaler.transform(X_fold_val)
    
    # SMOTE
    smote = SMOTE(random_state=42)
    X_resampled, y_resampled = smote.fit_resample(X_fold_train, y_fold_train)
    
    # Train
    lr_model.fit(X_resampled, y_resampled)
    
    # Predict
    val_preds = lr_model.predict(X_fold_val)
    
    # Metrics
    f1 = f1_score(y_fold_val, val_preds, average='weighted')
    acc = accuracy_score(y_fold_val, val_preds)
    
    LR_fold_f1.append(f1)
    LR_fold_acc.append(acc)
    
    print(f"\nFold {fold+1} - F1: {f1:.4f}, Accuracy: {acc:.4f}")
    # print(classification_report(y_fold_val, val_preds))

print(f"\nAverage F1 across folds: {np.mean(LR_fold_f1):.4f}")
print(f"Average Accuracy across folds: {np.mean(LR_fold_acc):.4f}")

# FINAL TRAIN ON FULL DATA

X_train_num = X_train.select_dtypes(include=[np.number])
X_test_num = X_test.select_dtypes(include=[np.number])

# Impute
final_imputer = SimpleImputer(strategy='median')
X_train_imputed = final_imputer.fit_transform(X_train_num)
X_test_imputed = final_imputer.transform(X_test_num)

# Scale
final_scaler = StandardScaler()
X_train_scaled = final_scaler.fit_transform(X_train_imputed)
X_test_scaled = final_scaler.transform(X_test_imputed)

# SMOTE
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_train_scaled, y_train)

# Train final model
lr_model.fit(X_resampled, y_resampled)


# TEST SET

test_preds = lr_model.predict(X_test_scaled)
test_probs = lr_model.predict_proba(X_test_scaled)

print("TEST PERFORMANCE")

if 'y_test' in locals():
    print(f"F1 Score: {f1_score(y_test, test_preds, average='weighted'):.4f}")
    print(f"Accuracy: {accuracy_score(y_test, test_preds):.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test, test_preds))

**Personlized Models**

In [ ]:
# checking distribution of stress labels across participants to understand class balance:
iOS_selected_features.groupby('uid')['stress'].value_counts(normalize=True)

counts = iOS_selected_features['uid'].value_counts()

print(counts.describe())
print("\nSmallest participants:")
print(counts.sort_values().head(10))

enough_data = counts[counts >= 30]
small_data = counts[counts < 30]
#Done to make sure that we have enough samples per participant for a 

print(f"Participants with >=30 samples: {len(enough_data)}")
print(f"Participants with <30 samples: {len(small_data)}")

In [ ]:
#Random Forest personalized models for each participant 

unique_participants = iOS_selected_features['uid'].unique()

personalized_results = {}

for participant in unique_participants:
    
    participant_data = iOS_selected_features[
        iOS_selected_features['uid'] == participant
    ]
    
    X_participant = participant_data.drop(columns=['stress', 'uid', 'day'])
    y_participant = participant_data['stress']
    
    # Skip if too little data
    if len(participant_data) < 10:
        continue
    
    # Skip if only one class
    if y_participant.nunique() < 2:
        continue
    
    # Train-test split
    X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(
        X_participant, y_participant, test_size=0.2, random_state=42
    )
    
    # Keep numeric features only
    X_train_p = X_train_p.select_dtypes(include=[np.number])
    X_test_p = X_test_p.select_dtypes(include=[np.number])
    
    # Drop columns that are all NaN in training set
    non_empty_cols = ~X_train_p.isna().all()

    X_train_p = X_train_p.loc[:, non_empty_cols]
    X_test_p = X_test_p.loc[:, non_empty_cols]
    
    # Impute missing values
    imputer = SimpleImputer(strategy='median')
    X_train_p = imputer.fit_transform(X_train_p)
    X_test_p = imputer.transform(X_test_p)
    
    # Random Forest model
    model = RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        min_samples_split=5,
        min_samples_leaf=2,
        max_features='sqrt',
        class_weight='balanced',  # helps imbalance per participant
        random_state=42,
        n_jobs=-1
    )
    
    # Train
    model.fit(X_train_p, y_train_p)
    
    # Predict
    preds = model.predict(X_test_p)
    
    # Metrics
    f1 = f1_score(y_test_p, preds, average='weighted')
    acc = accuracy_score(y_test_p, preds)
    
    personalized_results[participant] = {
        'f1': f1,
        'accuracy': acc
    }

# Averages
avg_f1 = np.mean([res['f1'] for res in personalized_results.values()])
avg_acc = np.mean([res['accuracy'] for res in personalized_results.values()])

print(f"\nAverage F1 across personalized RF models: {avg_f1:.4f}")
print(f"Average Accuracy across personalized RF models: {avg_acc:.4f}")

f1_scores = [res['f1'] for res in personalized_results.values()]
print("Min F1:", np.min(f1_scores))
print("Max F1:", np.max(f1_scores))

In [ ]:

# Creating personalized XGBoost models for each participant
unique_participants = iOS_selected_features['uid'].unique()
personalized_results = {}
skipped_participants = []
processed_participants = []



for participant in unique_participants:
    
    participant_data = iOS_selected_features[iOS_selected_features['uid'] == participant]
    
    X_participant = participant_data.drop(columns=['stress', 'uid', 'day'])
    y_participant = participant_data['stress']

    le = LabelEncoder()
    y_encoded = le.fit_transform(y_participant)

    if len(participant_data) < 10:
        skipped_participants.append((participant, "too few samples"))
        continue
    

    # Skip bad class distributions
    if min(np.bincount(y_encoded)) < 2:
        skipped_participants.append((participant, "class with <2 samples"))
        continue
    
    processed_participants.append(participant)
    

    # Train-test split
    X_train_p, X_test_p, y_train_adj, y_test_adj = train_test_split(
    X_participant,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded  #need to stratify to ensure all classes are represented in train/test splits
    )
    
    # Keep numeric only
    X_train_p = X_train_p.select_dtypes(include=[np.number])
    X_test_p = X_test_p.select_dtypes(include=[np.number])

    # Drop columns that are all NaN in training set
    non_empty_cols = ~X_train_p.isna().all()

    X_train_p = X_train_p.loc[:, non_empty_cols]
    X_test_p = X_test_p.loc[:, non_empty_cols]
    
    # Impute
    imputer = SimpleImputer(strategy='median')
    X_train_p = imputer.fit_transform(X_train_p)
    X_test_p = imputer.transform(X_test_p)
  
    # Model
    model = XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        objective='multi:softmax',
        # num_class = len(np.unique(y_encoded)),
        random_state=42,
        n_jobs=-1
    )
    
    #predict
    model.fit(X_train_p, y_train_adj)
    preds = model.predict(X_test_p)
    
    # Metrics
    f1 = f1_score(y_test_adj, preds, average='weighted')
    acc = accuracy_score(y_test_adj, preds)
    
    personalized_results[participant] = {
        'f1': f1,
        'accuracy': acc
    }

# Average performance
avg_f1 = np.mean([res['f1'] for res in personalized_results.values()])
avg_acc = np.mean([res['accuracy'] for res in personalized_results.values()])
print(f"Total participants: {len(unique_participants)}")
print(f"Processed participants: {len(processed_participants)}")
print(f"Skipped participants: {len(skipped_participants)}")

print("\nSkip reasons:")
for p, reason in skipped_participants[:10]:  # show first 10
    print(p, "-", reason)

print(f"\nAverage F1 across personalized models: {avg_f1:.4f}")
print(f"Average Accuracy across personalized models: {avg_acc:.4f}")
f1_scores = [res['f1'] for res in personalized_results.values()]
print("Min F1:", np.min(f1_scores))
print("Max F1:", np.max(f1_scores))

In [ ]:
#Multinomial Logistic Regression personalized models for each participant

unique_participants = iOS_selected_features['uid'].unique()

personalized_lr_results = {}

for participant in unique_participants:
    
    participant_data = iOS_selected_features[
        iOS_selected_features['uid'] == participant
    ]
    
    X_participant = participant_data.drop(columns=['stress', 'uid', 'day'])
    y_participant = participant_data['stress']
    
    # Skip checks
    if len(participant_data) < 10:
        continue
    
    if y_participant.nunique() < 2:
        continue
    
    # Split
    X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(
        X_participant, y_participant, test_size=0.2, random_state=42
    )
    
    # Numeric only
    X_train_p = X_train_p.select_dtypes(include=[np.number])
    X_test_p = X_test_p.select_dtypes(include=[np.number])
    
    # Drop columns that are all NaN in training set
    non_empty_cols = ~X_train_p.isna().all()

    X_train_p = X_train_p.loc[:, non_empty_cols]
    X_test_p = X_test_p.loc[:, non_empty_cols]

    # Impute
    imputer = SimpleImputer(strategy='median')
    X_train_p = imputer.fit_transform(X_train_p)
    X_test_p = imputer.transform(X_test_p)
    
    #  Scale 
    scaler = StandardScaler()
    X_train_p = scaler.fit_transform(X_train_p)
    X_test_p = scaler.transform(X_test_p)
    
    # Model
    model = LogisticRegression(
        solver='lbfgs',
        max_iter=1000,
        class_weight='balanced',  # helps within-user imbalance
        random_state=42
    )
    
    # Train
    model.fit(X_train_p, y_train_p)
    
    # Predict
    preds = model.predict(X_test_p)
    
    # Metrics
    f1 = f1_score(y_test_p, preds, average='weighted')
    acc = accuracy_score(y_test_p, preds)
    
    personalized_lr_results[participant] = {
        'f1': f1,
        'accuracy': acc
    }

# Averages
avg_f1 = np.mean([res['f1'] for res in personalized_lr_results.values()])
avg_acc = np.mean([res['accuracy'] for res in personalized_lr_results.values()])

print(f"\nAverage F1 across personalized LR models: {avg_f1:.4f}")
print(f"Average Accuracy across personalized LR models: {avg_acc:.4f}")

#  variability
f1_scores = [res['f1'] for res in personalized_lr_results.values()]
print("Min F1:", np.min(f1_scores))
print("Max F1:", np.max(f1_scores))